In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout
)

print("TensorFlow version:", tf.__version__)

In [ ]:

import pandas as pd


df = pd.read_csv('/content/heart.csv')


print("Dataset Shape:", df.shape)

print("\nFirst 5 Records:")
display(df.head())

print("\nDataset Information:")
df.info()


In [ ]:


print("Missing Values:")
print(df.isnull().sum())


print("\nNumber of Duplicate Rows:")
print(df.duplicated().sum())


print("\nStatistical Summary:")
display(df.describe())


df = df.drop_duplicates().copy()

print("\nDataset Shape After Removing Duplicates:")
print(df.shape)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


print("Target Distribution:")
print(df["target"].value_counts())


plt.figure(figsize=(7, 5))

sns.countplot(data=df, x="target")

plt.title("Heart Disease Target Distribution")
plt.xlabel("Target (0 = Negative, 1 = Positive)")
plt.ylabel("Number of Records")

plt.show()

In [ ]:

X = df.drop("target", axis=1)
y = df["target"]


print("Input Features Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nInput Features:")
display(X.head())

print("\nTarget Values:")
print(y.head())

In [ ]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
        y,
            test_size=0.2,
                random_state=42,
                    stratify=y
                    )


print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)
print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler


scaler = StandardScaler()


X_train_scaled = scaler.fit_transform(X_train)


X_test_scaled = scaler.transform(X_test)


print("Scaled Training Data:", X_train_scaled.shape)
print("Scaled Testing Data:", X_test_scaled.shape)

print("\nFirst 5 Scaled Records:")
print(X_train_scaled[:5])

In [ ]:

X_train_cnn = X_train_scaled.reshape(
    X_train_scaled.shape[0], 13, 1
    )


X_test_cnn = X_test_scaled.reshape(
    X_test_scaled.shape[0], 13, 1
        )


print("CNN Training Data Shape:", X_train_cnn.shape)
print("CNN Testing Data Shape:", X_test_cnn.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout


model = Sequential([
    Input(shape=(13, 1)),
    Conv1D(filters=32, kernel_size=3, activation='relu', padding='same'),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
    Flatten(),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])


model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping


early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)


history = model.fit(
    X_train_cnn,
    y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

print("CNN model training completed!")

In [ ]:

import matplotlib.pyplot as plt


plt.figure(figsize=(12, 5))


plt.subplot(1, 2, 1)

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.title("CNN Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()


plt.subplot(1, 2, 2)

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("CNN Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


test_loss, test_accuracy = model.evaluate(
    X_test_cnn,
    y_test,
    verbose=0
)

print("Test Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_accuracy * 100, 2), "%")


y_probability = model.predict(X_test_cnn, verbose=0)
y_pred = (y_probability >= 0.5).astype(int).ravel()


print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Target 0", "Target 1"],
        zero_division=0
    )
)


cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Target 0", "Target 1"],
    yticklabels=["Target 0", "Target 1"]
)

plt.title("Heart Disease Prediction - Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd


sample = pd.DataFrame([{
    "age": 55,
    "sex": 1,
    "cp": 0,
    "trestbps": 140,
    "chol": 250,
    "fbs": 0,
    "restecg": 1,
    "thalach": 150,
    "exang": 0,
    "oldpeak": 1.0,
    "slope": 1,
    "ca": 0,
    "thal": 2
}])


sample_scaled = scaler.transform(sample)


sample_cnn = sample_scaled.reshape(1, 13, 1)



probability = model.predict(sample_cnn, verbose=0)[0][0]
predicted_class = int(probability >= 0.5)

print("Predicted Class:", predicted_class)
print(f"Model Output Score: {probability:.4f}")

if predicted_class == 1:
    print("The model predicts Class 1.")
else:
    print("The model predicts Class 0.")

In [ ]:
import joblib


model.save("/content/heart_disease_cnn.keras")


joblib.dump(scaler, "/content/heart_disease_scaler.pkl")

print("CNN model and scaler saved successfully!")